## 0)  Project Root

In [1]:
import sys
from pathlib import Path

def get_project_root(project_dir_name="Project", marker=".git"):
    """
    1) Walk upwards until we find the repo root (contains .git).
    2) Return <repo_root>/<project_dir_name> as the actual project root
       (the folder that contains models/, training/, notebooks/, etc.).
    3) Add that project root to sys.path for imports.
    """
    current = Path.cwd().resolve()

    # Step 1: find repo root by .git
    repo_root = None
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            repo_root = parent
            break

    if repo_root is None:
        raise RuntimeError(f"Repo root not found (no '{marker}' directory)")

    # Step 2: define actual project root
    project_root = repo_root / project_dir_name
    if not project_root.exists():
        raise RuntimeError(
            f"Found repo root at {repo_root}, but '{project_dir_name}' folder not found."
        )

    # Step 3: add to sys.path
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f"[OK] Repo root:    {repo_root}")
    print(f"[OK] Project root: {project_root}")
    return project_root

PROJECT_ROOT = get_project_root()



[OK] Repo root:    /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning
[OK] Project root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project


In [4]:
import torch

# -------------------------------------------------------------
# Device selection (CPU / CUDA / MPS)
# -------------------------------------------------------------
# We automatically pick the "best" available device:
#   1) Apple Silicon GPU (MPS) if available
#   2) NVIDIA GPU (CUDA) if available
#   3) Fallback to CPU otherwise
#
# This way the same code runs efficiently on different machines.
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using device: CUDA")
else:
    device = torch.device("cpu")
    print("Using device: CPU")

"""
# -------------------------------------------------------------
# Basic shape information (for sanity + model config)
# -------------------------------------------------------------
# X_train_scaled has shape: (num_train_samples, seq_len, num_features)
# We read seq_len and num_features directly from the data to avoid
# hard-coding them.
# -------------------------------------------------------------
seq_len = X_train_scaled.shape[1]
num_features = X_train_scaled.shape[2]
print(f"Sequence length: {seq_len}, num_features: {num_features}")
"""

Using device: MPS (Apple Silicon GPU)


'\n# -------------------------------------------------------------\n# Basic shape information (for sanity + model config)\n# -------------------------------------------------------------\n# X_train_scaled has shape: (num_train_samples, seq_len, num_features)\n# We read seq_len and num_features directly from the data to avoid\n# hard-coding them.\n# -------------------------------------------------------------\nseq_len = X_train_scaled.shape[1]\nnum_features = X_train_scaled.shape[2]\nprint(f"Sequence length: {seq_len}, num_features: {num_features}")\n'

In [5]:
from pathlib import Path
import json
import torch


SEQ_DIR = PROJECT_ROOT / "03_Sequences"
RUN_DIR = PROJECT_ROOT / "runs" / "LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC"  # <- pick one


CONFIG_PATH = RUN_DIR / "config.json"
CKPT_PATH   = RUN_DIR / "best_model.pt"


In [7]:
from utils.data_utils import make_test_loader

test_loader, X_test, y_test, config = make_test_loader(
    run_dir=RUN_DIR,
    seq_dir=SEQ_DIR,
    shuffle=False
)

model_class_name = config["model_class"]
model_kwargs     = config["model_kwargs"]
seq_len = config["seq_len"]
SEQ_LEAF_DIR = SEQ_DIR / f"seq{seq_len}"   # SEQ_DIR is base: .../03_Sequences

##################################################################################

## TESTING (Legacy)

In [11]:
from models import LSTMClassifier
import hashlib

# -------------------------------------------------------------
# Load the best-performing model checkpoint
# -------------------------------------------------------------
# During training, we saved the model weights (state_dict) every
# time the validation loss improved.  This file contains ONLY the
# trained parameters — not the model class itself.
#
# To load it correctly, we must:
#   1. Recreate the SAME model architecture (same layers/sizes)
#   2. Load the saved state_dict into that model
#   3. Move the model to the correct device (CPU / GPU / MPS)
#
# NOTE:
#   • If ANY model hyperparameter changes (num_features, hidden_size,
#     num_layers, dropout), loading will fail.
#   • This pattern is standard in PyTorch: checkpoint = weights only.
# -------------------------------------------------------------

# -------------------------------------------------------------
# Manually select run folder inside /runs
# -------------------------------------------------------------
RUN_NAME = "LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC"  # <-- EDIT THIS

RUN_DIR = PROJECT_ROOT / "runs" / RUN_NAME
assert RUN_DIR.exists(), f"RUN_DIR not found: {RUN_DIR}"

print("Using RUN_DIR:", RUN_DIR)



best_model = LSTMClassifier(
    input_size=15, hidden_size=64, num_layers=2, dropout=0.1
)

CKPT_PATH = Path(RUN_DIR) / "best_model.pt"
assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"

# Load weights from file into the model
best_model.load_state_dict(
    torch.load(CKPT_PATH, map_location=device)
)

# Ensure the model runs on the same device as the evaluation tensors
best_model.to(device)

def fp(m):
    h = hashlib.sha256()
    for k in sorted(m.state_dict().keys()):
        t = m.state_dict()[k].detach().cpu().contiguous().numpy()
        h.update(k.encode()); h.update(t.tobytes())
    return h.hexdigest()[:16]

print("MODEL_FP:", fp(best_model))

Using RUN_DIR: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC
MODEL_FP: 6eae0c985503c3f6


In [13]:
meta = json.loads((SEQ_LEAF_DIR / "meta.json").read_text())

In [15]:
# -------------------------------------------------------------
# 2) Accuracy + confusion per t_to_end_min
# -------------------------------------------------------------
# Here we go beyond global metrics and analyze performance
# as a function of "minutes to end of 15-min window".
#
# Steps:
#   1) Find where t_to_end_min lives in the feature vector
#   2) Extract t_to_end_min for each TEST sample
#      (from the *unscaled* X_test, last time step in each sequence)
#   3) Run a forward pass over test_loader to collect:
#         - predicted labels
#         - true labels
#   4) Build a DataFrame and compute:
#         - TP/FP/TN/FN per t_to_end_min
#         - accuracy per t_to_end_min
# -------------------------------------------------------------
import pandas as pd
from tqdm.auto import tqdm
import numpy as np

# 2.1) Find the feature index for t_to_end_min
t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
print(f"t_to_end_min is at feature index: {t_to_end_min_idx}")

# 2.2) Extract t_to_end_min from the UN-SCALED test data
#      Each sequence has shape (seq_len, num_features).
#      We take the LAST timestep [-1] for each sample:
#         → this corresponds to the "current" minute the model is predicting for.
t_to_end_min_values = X_test[:, -1, t_to_end_min_idx]  # shape: (num_test,)

print(f"Unique t_to_end_min values: {np.unique(t_to_end_min_values)}")

# 2.3) Collect predictions and true labels from the best model
best_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_i, (X_batch, y_batch) in enumerate(
        tqdm(test_loader, desc="Predicting (test)", leave=False)
    ):
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1).to(device)

        logits = best_model(X_batch).view(-1)     # IMPORTANT: use best_model
        probs  = torch.sigmoid(logits)
        preds  = (probs >= 0.5).long()

        if batch_i == 0:
            xb = X_batch.detach().cpu().contiguous().numpy()
            lg = logits.detach().cpu().contiguous().numpy()
            pr = preds.detach().cpu().contiguous().numpy()

            def fp(m):
                h = hashlib.sha256()
                for k in sorted(m.state_dict().keys()):
                    t = m.state_dict()[k].detach().cpu().contiguous().numpy()
                    h.update(k.encode()); h.update(t.tobytes())
                return h.hexdigest()[:16]

            print("MODEL_FP:", fp(best_model))
            print("X_FP:", hashlib.sha256(xb.tobytes()).hexdigest()[:16])
            print("LOGITS_FP:", hashlib.sha256(lg.tobytes()).hexdigest()[:16])
            print("PREDS_FP:", hashlib.sha256(pr.tobytes()).hexdigest()[:16])

            print("training:", best_model.training)
            print("param dtype:", next(best_model.parameters()).dtype)
            print("X dtype:", X_batch.dtype)
            print("device:", next(best_model.parameters()).device)

        # IMPORTANT: extend with numpy arrays, not torch tensors
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_preds  = np.array(all_preds, dtype=int).reshape(-1)
all_labels = np.array(all_labels, dtype=int).reshape(-1)

assert (
    all_preds.shape[0] == t_to_end_min_values.shape[0]
), "Mismatch: number of test predictions != number of t_to_end_min entries"


# 2.4) Build a DataFrame for per-minute analysis
df_results = pd.DataFrame(
    {
        "t_to_end_min": t_to_end_min_values,
        "y_true": all_labels,
        "y_pred": all_preds,
    }
)

# Add confusion components per sample
df_results["tp"] = ((df_results.y_true == 1) & (df_results.y_pred == 1)).astype(int)
df_results["fp"] = ((df_results.y_true == 0) & (df_results.y_pred == 1)).astype(int)
df_results["tn"] = ((df_results.y_true == 0) & (df_results.y_pred == 0)).astype(int)
df_results["fn"] = ((df_results.y_true == 1) & (df_results.y_pred == 0)).astype(int)
df_results["correct"] = (df_results["tp"] + df_results["tn"]).astype(int)

# 2.5) Aggregate stats by t_to_end_min
stats = (
    df_results.groupby("t_to_end_min")
    .agg(
        count=("correct", "count"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        tn=("tn", "sum"),
        fn=("fn", "sum"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
)

stats["accuracy_pct"] = stats["accuracy"] * 100

# 2.6) Print per-minute results
print("\n" + "=" * 70)
print("ACCURACY + CONFUSION METRICS PER t_to_end_min")
print("=" * 70)
print(stats.to_string(index=False))
print("=" * 70)

# 2.7) Baseline vs model
# Baseline (always predicting 1) accuracy = fraction of positives in test labels
baseline_acc = all_labels.mean()
model_acc = (all_preds == all_labels).mean()

print(f"\nBaseline Accuracy (always predict 1): {baseline_acc:.4f}")
print(f"Model Test Accuracy (from preds):      {model_acc:.4f}")

t_to_end_min is at feature index: 14
Unique t_to_end_min values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]


Predicting (test):   0%|          | 0/29592 [00:00<?, ?it/s]

MODEL_FP: 6eae0c985503c3f6
X_FP: bafdebb3d1c993b8
LOGITS_FP: 8a4b4452a9b2f74a
PREDS_FP: 76c862828aa3b269
training: False
param dtype: torch.float32
X dtype: torch.float32
device: mps:0

ACCURACY + CONFUSION METRICS PER t_to_end_min
 t_to_end_min  count    tp   fp    tn   fn  accuracy  accuracy_pct
          1.0  31564 15370  364 15459  371  0.976714     97.671398
          2.0  31564 14768 1073 14750  973  0.935179     93.517932
          3.0  31564 14360 1700 14123 1381  0.902389     90.238880
          4.0  31564 13963 2157 13666 1778  0.875333     87.533266
          5.0  31564 13588 2719 13104 2153  0.845647     84.564694
          6.0  31564 13276 3200 12623 2465  0.820523     82.052338
          7.0  31564 13054 3686 12137 2687  0.798093     79.809276
          8.0  31564 12734 4226 11597 3007  0.770847     77.084653
          9.0  31564 12394 4837 10986 3347  0.740717     74.071727
         10.0  31564 12054 5555 10268 3687  0.707198     70.719807
         11.0  31564 11851 6239

In [30]:
def hash_array(a: np.ndarray) -> str:
    return hashlib.md5(a.tobytes()).hexdigest()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("X_test hash:", hash_array(X_test))
print("y_test hash:", hash_array(y_test))

X_test shape: (473458, 60, 15)
y_test shape: (473458,)
X_test hash: 821e37fd2e30d0dd30749865b5d2ab94
y_test hash: 7138a0d4459944e41259e786b11688ec
